# Production Workflow Agent
### Manufacturing AI/ML Series — Playbook 4 | ForgeFlow Reference Platform

**What this notebook builds:**  
A production workflow agent that reconstructs current shop floor state from work order and job card data, applies risk rules to detect overdue WOs and bottlenecks, and generates a daily production briefing.

**Architecture:** Sense → Reason → Act
- **Sense:** Read work orders and job cards to reconstruct current production state
- **Reason:** Apply calibrated rules to detect overdue WOs, workstation bottlenecks, delivery risk
- **Act:** Generate structured briefing — rule-based core, optional LLM natural language extension

**Core analysis:** No API key required  
**LLM extension (Section 8):** Requires `ANTHROPIC_API_KEY`

---
**Data source:** [ForgeFlow Reference Platform](https://github.com/Nub-Labs/forgeflow-reference-platform)  
**Playbook:** https://nublabs.com/playbooks/manufacturing/production-workflow-agent

## 1. Setup

In [ ]:
!pip install pandas matplotlib requests --quiet
# anthropic is only needed for the optional LLM section
# !pip install anthropic --quiet

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import requests
from datetime import date, timedelta
import json

BASE = 'https://raw.githubusercontent.com/Nub-Labs/forgeflow-reference-platform/main/data'

def load(path):
    r = requests.get(f'{BASE}/{path}')
    r.raise_for_status()
    return pd.DataFrame(r.json())

print('Setup complete.')

## 2. Load data

In [ ]:
work_orders = load('production/work_orders.json')
job_cards   = load('production/job_cards.json')
items       = load('master/items.json')
sales_orders = load('sales/sales_orders.json')

print(f'Work orders: {len(work_orders)}')
print(f'Job cards:   {len(job_cards)}')
print()
work_orders.head(3)

## 3. Understand the data structure

Each work order has 5 job cards (one per operation). The routing sequence is fixed:

```
Induction Heating → Die Forging → Flash Trimming → Dimensional Inspection → CNC Turning / VMC Milling (optional)
```

The agent determines the **current operation** for each WO by finding the last completed job card in the routing sequence.

In [ ]:
print('Operations in job cards:')
print(job_cards['operation'].value_counts().to_string())
print()
print('Workstations:')
print(job_cards['workstation'].value_counts().to_string())
print()
print('Status breakdown:')
print(job_cards['status'].value_counts().to_string())

## 4. Load calibrated lead times (from Playbook 2)

Playbook 2 (Production Planning Assistant) established that the actual median lead time for most item families is 18 days, regardless of the planned template value. We use these calibrated values as the overdue threshold.

In [ ]:
# Calibrated lead times per item family (from Playbook 2)
# 'calibrated_days' = recommended planned lead time based on historical actuals
calibration = pd.DataFrame([
    {'item_group': 'FG-FL', 'family': 'Flange',          'calibrated_days': 20, 'press': 'Heavy'},
    {'item_group': 'FG-CR', 'family': 'Connecting Rod',  'calibrated_days': 20, 'press': 'Heavy'},
    {'item_group': 'FG-YK', 'family': 'Yoke',            'calibrated_days': 20, 'press': 'Heavy'},
    {'item_group': 'FG-CS', 'family': 'Crankshaft',      'calibrated_days': 20, 'press': 'Heavy'},
    {'item_group': 'FG-GB', 'family': 'Gear Blank',      'calibrated_days': 20, 'press': 'Heavy'},
    {'item_group': 'FG-SP', 'family': 'Steering Spindle','calibrated_days': 20, 'press': 'Heavy'},
    {'item_group': 'FG-CP', 'family': 'Coupling Half',   'calibrated_days': 20, 'press': 'Heavy'},
    {'item_group': 'FG-KP', 'family': 'Knuckle Pin',     'calibrated_days': 20, 'press': 'Heavy'},
    {'item_group': 'FG-VC', 'family': 'Valve Component', 'calibrated_days': 20, 'press': 'Light'},
])

# Add item_prefix for joining
work_orders['item_prefix'] = work_orders['item_code'].str[:5]

print('Calibration table loaded.')
calibration

## 5. SENSE — Reconstruct current production state

In a live system, some job cards would be In Progress or Pending. Since the ForgeFlow data contains only Completed WOs, we simulate an in-progress snapshot by selecting a subset of WOs and computing their state at a mid-period date.

The agent logic works identically on live ERP data.

In [ ]:
# Routing sequence (order matters)
ROUTING = [
    'Induction Heating',
    'Die Forging',
    'Flash Trimming',
    'Dimensional Inspection',
    'CNC Turning',
    'VMC Milling',
]

def routing_index(op):
    for i, r in enumerate(ROUTING):
        if op.startswith(r[:6]):  # prefix match handles variants
            return i
    return 99

job_cards['routing_idx'] = job_cards['operation'].apply(routing_index)

# Simulate in-progress: take WOs from Q1, assume snapshot date = 2026-03-01
# Mark operations completed if routing_idx <= a simulated 'stage reached'
import random
random.seed(42)

# Select 10 WOs for the agent snapshot
snapshot_wos = work_orders.sample(10, random_state=42).copy()
snapshot_wos['actual_start_date'] = pd.to_datetime(snapshot_wos['actual_start_date'])

# Assign 'current stage' (0-5): how many operations are done
snapshot_wos['ops_completed'] = [random.randint(1, 4) for _ in range(len(snapshot_wos))]

# Calculate elapsed days from actual start to today
today = pd.Timestamp('2026-07-16')
snapshot_wos['elapsed_days'] = ((today - snapshot_wos['actual_start_date']).dt.days).clip(lower=1)

# Join calibration
snapshot_wos = snapshot_wos.merge(calibration[['item_group', 'calibrated_days', 'press']], 
                                   left_on='item_prefix', right_on='item_group', how='left')
snapshot_wos['calibrated_days'] = snapshot_wos['calibrated_days'].fillna(20)

# Current and next operation from routing
snapshot_wos['current_op'] = snapshot_wos['ops_completed'].apply(
    lambda n: ROUTING[n] if n < len(ROUTING) else 'Complete'
)
snapshot_wos['workstation'] = snapshot_wos['current_op'].map({
    'Induction Heating': 'Induction Furnace Bay',
    'Die Forging': 'Heavy Forging Press',
    'Flash Trimming': 'Heavy Forging Press',
    'Dimensional Inspection': 'Quality Lab',
    'CNC Turning': 'CNC Turning Bay',
    'VMC Milling': 'VMC Bay',
    'Complete': '-',
})

print('Production snapshot reconstructed:')
print(snapshot_wos[['wo_id', 'item_name', 'elapsed_days', 'calibrated_days', 'current_op', 'workstation']].to_string(index=False))

## 6. REASON — Apply risk rules

In [ ]:
def classify_status(row):
    if row['elapsed_days'] > row['calibrated_days']:
        return 'OVERDUE'
    elif row['elapsed_days'] > row['calibrated_days'] * 0.75:
        return 'AT_RISK'
    else:
        return 'ON_TRACK'

snapshot_wos['status'] = snapshot_wos.apply(classify_status, axis=1)
snapshot_wos['overrun_days'] = (snapshot_wos['elapsed_days'] - snapshot_wos['calibrated_days']).clip(lower=0)

# Workstation load
active = snapshot_wos[snapshot_wos['current_op'] != 'Complete']
ws_load = active.groupby('workstation').size().reset_index(name='concurrent_wos').sort_values('concurrent_wos', ascending=False)
bottleneck_ws = ws_load.iloc[0]['workstation'] if len(ws_load) > 0 else 'None'
bottleneck_count = ws_load.iloc[0]['concurrent_wos'] if len(ws_load) > 0 else 0

print('Status summary:')
print(snapshot_wos['status'].value_counts().to_string())
print()
print('Workstation load:')
print(ws_load.to_string(index=False))
print()
print(f'Primary bottleneck: {bottleneck_ws} ({bottleneck_count} concurrent WOs)')

## 7. ACT — Generate structured daily briefing

In [ ]:
overdue = snapshot_wos[snapshot_wos['status'] == 'OVERDUE']
at_risk = snapshot_wos[snapshot_wos['status'] == 'AT_RISK']
on_track = snapshot_wos[snapshot_wos['status'] == 'ON_TRACK']

# Build structured briefing JSON
briefing = {
    'date': '2026-07-16',
    'generated_at': '07:30',
    'summary': {
        'total_open_wos': len(snapshot_wos),
        'overdue': len(overdue),
        'at_risk': len(at_risk),
        'on_track': len(on_track),
        'primary_bottleneck': bottleneck_ws,
        'bottleneck_concurrent_wos': int(bottleneck_count),
    },
    'flags': [],
    'recommended_actions': [],
}

for _, row in overdue.iterrows():
    briefing['flags'].append({
        'type': 'OVERDUE',
        'wo_id': row['wo_id'],
        'item': row['item_name'],
        'elapsed_days': int(row['elapsed_days']),
        'planned_days': int(row['calibrated_days']),
        'overrun_days': int(row['overrun_days']),
        'current_op': row['current_op'],
        'workstation': row['workstation'],
    })

for _, row in at_risk.iterrows():
    briefing['flags'].append({
        'type': 'AT_RISK',
        'wo_id': row['wo_id'],
        'item': row['item_name'],
        'elapsed_days': int(row['elapsed_days']),
        'planned_days': int(row['calibrated_days']),
    })

briefing['recommended_actions'] = [
    f"Escalate {len(overdue)} overdue WOs to production supervisor before first shift",
    f"Check {bottleneck_ws} queue — {bottleneck_count} concurrent WOs, primary constraint",
]
if len(at_risk) > 0:
    briefing['recommended_actions'].append(f"{len(at_risk)} WO(s) at risk — review with planning before midday")

print('=== DAILY PRODUCTION BRIEFING ===')
print(json.dumps(briefing, indent=2))

## 7b. Visualise production state

In [ ]:
status_colors = {'ON_TRACK': '#10b981', 'AT_RISK': '#f59e0b', 'OVERDUE': '#ef4444'}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('ForgeFlow Production Workflow Agent — Daily Snapshot 16 Jul 2026', fontsize=12, fontweight='bold')

# Left: elapsed vs calibrated per WO
wos = snapshot_wos.sort_values('elapsed_days', ascending=True)
y_pos = range(len(wos))
colors = [status_colors[s] for s in wos['status']]
short_names = wos['item_name'].str[:25]

ax1.barh(y_pos, wos['calibrated_days'], color='#e2e8f0', label='Calibrated plan')
ax1.barh(y_pos, wos['elapsed_days'], color=colors, alpha=0.85, label='Elapsed days')
ax1.set_yticks(y_pos)
ax1.set_yticklabels(short_names, fontsize=8)
ax1.set_xlabel('Days')
ax1.set_title('Elapsed vs calibrated lead time', fontsize=10)
ax1.axvline(x=20, color='#64748b', linestyle='--', linewidth=0.8, label='20-day target')

legend_patches = [mpatches.Patch(color=c, label=s) for s, c in status_colors.items()]
ax1.legend(handles=legend_patches, loc='lower right', fontsize=8)

# Right: workstation load
if len(ws_load) > 0:
    ax2.bar(ws_load['workstation'], ws_load['concurrent_wos'],
            color=['#ef4444' if ws == bottleneck_ws else '#3b82f6' for ws in ws_load['workstation']])
    ax2.set_title('Concurrent WOs per workstation', fontsize=10)
    ax2.set_ylabel('Active work orders')
    plt.setp(ax2.get_xticklabels(), rotation=30, ha='right', fontsize=8)
    ax2.axhline(y=2, color='#f59e0b', linestyle='--', linewidth=0.8, label='Bottleneck threshold')
    ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig('production_workflow_agent.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved.')

---
## 8. Optional — LLM Extension (Anthropic API)

This section passes the structured briefing JSON to Claude and asks it to write a natural language summary for the production manager. 

**Requires:** `ANTHROPIC_API_KEY` set as a Colab secret or environment variable.

The rule-based sections above are complete without this step. Add the LLM layer only once the rule-based output is validated.

In [ ]:
import os

ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY', '')

if not ANTHROPIC_API_KEY:
    print('No ANTHROPIC_API_KEY found. Skipping LLM section.')
    print('To enable: set ANTHROPIC_API_KEY in Colab Secrets (key icon in left sidebar).')
else:
    print('API key found. Proceeding with LLM extension.')

In [ ]:
if ANTHROPIC_API_KEY:
    import anthropic

    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

    system_prompt = """You are a production operations assistant for a precision engineering manufacturer.
You receive a structured JSON briefing from a production monitoring agent each morning.
Write a concise, plain-English summary (under 150 words) that a production manager can act on immediately.
Lead with the most critical issue. Name specific work orders and workstations.
End with the top two recommended actions for today.
Do not use headers or bullet points — write it as a single briefing paragraph followed by two numbered actions."""

    user_message = f"""Production briefing data for 16 Jul 2026:

{json.dumps(briefing, indent=2)}

Write the daily briefing for the production manager."""

    response = client.messages.create(
        model='claude-sonnet-5',
        max_tokens=300,
        messages=[{'role': 'user', 'content': user_message}],
        system=system_prompt,
    )

    print('=== LLM-GENERATED DAILY BRIEFING ===')
    print()
    print(response.content[0].text)
else:
    print('(LLM section skipped — no API key)')
    print()
    print('Example output (what the LLM would produce):')
    print()
    example = (
        f"{briefing['summary']['overdue']} work orders are past their calibrated 20-day lead time as of today. "
        f"The {bottleneck_ws} is carrying {bottleneck_count} concurrent WOs and is the active bottleneck. "
        f"Review the overdue WOs with the production supervisor before the first shift starts "
        f"and confirm whether any customer delivery dates are at risk.\n\n"
        f"1. Check {bottleneck_ws} queue and prioritise the highest-overrun WOs.\n"
        f"2. Contact planning to review calibrated template updates for legacy single-day plans."
    )
    print(example)

---
## 9. Production architecture: taking this to a scheduled job

```
┌─────────────────────────────────────────────────────────────────┐
│  Scheduled job (06:30 daily)                                    │
│                                                                 │
│  1. Pull open WOs + job cards from ERP API or nightly export   │
│  2. Reconstruct current stage per WO (sense)                   │
│  3. Apply risk rules: overdue, bottleneck, delivery risk (reason)│
│  4. Generate structured JSON briefing (act)                     │
│  5. [Optional] Call LLM for natural language summary            │
│  6. Send to Slack / email / ERP task                           │
│                                                                 │
│  Next iteration (after 4 weeks of validated read-only output): │
│  - Give agent ERP write tools                                  │
│  - Auto-create task escalations for overdue WOs                │
│  - Auto-update planned dates on legacy templates               │
│  - Draft customer communications (human approves before send)  │
└─────────────────────────────────────────────────────────────────┘
```

**Rule for adding write access:** validate read-only output for 4 weeks. Only add tools that the agent uses correctly in at least 90% of read-only runs. Never auto-send customer messages without human approval.

---
## Series complete

This is the final playbook in the Manufacturing AI/ML series on ForgeFlow:

| Playbook | Technique | Output |
|---|---|---|
| 1. AI Procurement Assistant | Composite scoring | Supplier grades + reorder recommendations |
| 2. Production Planning Assistant | Lead time calibration | Corrected WO templates, bottleneck identification |
| 3. Supplier Intelligence Dashboard | Spend concentration + HHI | Risk-adjusted spend table, sole-source flags |
| 4. Production Workflow Agent (this) | Sense-reason-act loop + LLM | Daily production briefing, delivery risk alerts |

**More series at:** https://nublabs.com/playbooks